# GeoFire-X CPU certification: lot-005
Private CPU only. No Internet, installation, training or GPU. Missing dependency or checksum mismatch means STOP.


In [ ]:
import json, os
from pathlib import Path
namespace = {'__name__': 'geofirex_cpu_helpers'}
exec(compile('"""CPU-only Kaggle input helpers; bounded streaming, no network or training."""\nimport hashlib\nimport importlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport sys\nimport time\nimport zipfile\nimport zlib\n\nBLOCK = 1024*1024\n\n\ndef sha_file(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(BLOCK), b\'\'): h.update(block)\n    return h.hexdigest()\n\n\ndef safe_child(root, name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or \'..\' in p.parts or \'\\\\\' in name or \':\' in name:\n        raise ValueError(\'Unsafe relative path\')\n    target = Path(root)/name\n    if not target.resolve().is_relative_to(Path(root).resolve()):\n        raise ValueError(\'Path escapes root\')\n    return target\n\n\ndef bootstrap_code(source, work, payload_sha, manifest_sha):\n    source, work = Path(source), Path(work)\n    if work.exists(): raise FileExistsError(\'New working directory required\')\n    payload = source/\'payload.zip\'\n    archive = None\n    try:\n        if payload.is_file():\n            if sha_file(payload) != payload_sha: raise ValueError(\'Code ZIP SHA mismatch\')\n            archive = zipfile.ZipFile(payload)\n            names = archive.namelist()\n            manifest_bytes = archive.read(\'code-manifest.json\')\n            manifest = json.loads(manifest_bytes)\n            if len(names) != len(set(names)) or set(names) != set(manifest) | {\'code-manifest.json\'}:\n                raise ValueError(\'Code ZIP inventory differs\')\n        else:\n            matches = list(source.rglob(\'code-manifest.json\'))\n            if len(matches) != 1: raise ValueError(\'Expected one expanded code manifest\')\n            source = matches[0].parent\n            manifest_bytes = matches[0].read_bytes()\n            manifest = json.loads(manifest_bytes)\n        if hashlib.sha256(manifest_bytes).hexdigest() != manifest_sha:\n            raise ValueError(\'Pinned code manifest SHA mismatch\')\n        work.mkdir(parents=True, exist_ok=False)\n        for name, sha in manifest.items():\n            dest = safe_child(work, name)\n            origin = None if archive else safe_child(source, name)\n            if origin and origin.is_symlink(): raise ValueError(\'Symlink refused\')\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            h = hashlib.sha256()\n            with (archive.open(name) if archive else origin.open(\'rb\')) as inp, dest.open(\'xb\') as out:\n                for block in iter(lambda: inp.read(BLOCK), b\'\'): h.update(block); out.write(block)\n            if h.hexdigest() != sha: raise ValueError(\'Expanded code file SHA mismatch: \'+name)\n        (work/\'code-manifest.json\').write_bytes(manifest_bytes)\n        return work\n    finally:\n        if archive: archive.close()\n\n\ndef check_runtime(work, bundle_relative):\n    if sys.version_info < (3, 11): raise RuntimeError(\'Operational preflight requires Python >=3.11\')\n    os.environ.update(CUDA_VISIBLE_DEVICES=\'\', OMP_NUM_THREADS=\'2\', MKL_NUM_THREADS=\'2\', OPENBLAS_NUM_THREADS=\'2\')\n    sys.path.insert(0, str(Path(work)/\'src\'))\n    versions, missing = {}, []\n    for module, distribution in [(\'numpy\',\'numpy\'), (\'rasterio\',\'rasterio\'), (\'torch\',\'torch\'), (\'mlflow\',\'mlflow\'),\n                                 (\'yaml\',\'pyyaml\'), (\'matplotlib\',\'matplotlib\'), (\'sklearn\',\'scikit-learn\'),\n                                 (\'pyarrow\',\'pyarrow\'), (\'psutil\',\'psutil\')]:\n        try:\n            importlib.import_module(module)\n            versions[distribution] = importlib.metadata.version(distribution)\n        except ImportError: missing.append(distribution)\n    if missing: raise RuntimeError(\'Missing dependencies; STOP, no automatic installation: \'+\',\'.join(missing))\n    psutil_version = tuple(int(v) for v in versions[\'psutil\'].split(\'.\')[:2])\n    if not (psutil_version >= (5,9) and psutil_version < (8,0)):\n        raise RuntimeError(\'psutil outside reviewed >=5.9,<8 range\')\n    import pyarrow.parquet as pq\n    bundle = Path(work)/bundle_relative\n    parquet_rows = {name: pq.read_table(bundle/f\'metadata/{name}.parquet\').num_rows for name in (\'observations\',\'windows\')}\n    norm_path = bundle/\'metadata/normalization.json\'\n    norm = json.loads(norm_path.read_text())\n    if sha_file(norm_path) != \'aa30df4e3d6d895760f04e35d74bcbbac0d739cea045d236517401f5f9f4c7f6\':\n        raise ValueError(\'Train normalizer SHA mismatch\')\n    provenance = norm[\'provenance\']\n    if (provenance[\'train_events\'], provenance[\'validation_inputs\'], provenance[\'test_inputs\']) != (250,0,0):\n        raise ValueError(\'Train-only provenance differs\')\n    working_free=shutil.disk_usage(\'/kaggle/working\').free\n    scratch=Path(\'/kaggle/temp\'); scratch.mkdir(exist_ok=True)\n    scratch_free=shutil.disk_usage(scratch).free\n    return dict(python=sys.version, versions=versions, parquet_rows=parquet_rows,\n                normalizer_sha256=sha_file(norm_path), train_events=250, validation_fit_inputs=0, test_fit_inputs=0,\n                working_free_bytes=working_free, scratch_free_bytes=scratch_free,\n                disk_gate_for_future_training=working_free >= 20_000_000_000 and scratch_free >= 40_000_000_000)\n\n\ndef resource_check(deadline, min_available=2*1024**3):\n    if time.monotonic() > deadline: raise TimeoutError(\'CPU certification deadline reached\')\n    mem = Path(\'/proc/meminfo\')\n    if mem.exists():\n        available = next(int(row.split()[1])*1024 for row in mem.read_text().splitlines() if row.startswith(\'MemAvailable:\'))\n        if available < min_available: raise MemoryError(\'CPU certification memory reserve reached\')\n\n\ndef certify_lot(canonical_path, dataset_root, out, *, pinned_identity, seconds=3600, inspect_geotiff=True):\n    lot = json.loads(Path(canonical_path).read_text())\n    unsigned = {k: v for k, v in lot.items() if k != \'identity\'}\n    identity = hashlib.sha256(json.dumps(unsigned, sort_keys=True, separators=(\',\', \':\')).encode()).hexdigest()\n    if identity != lot[\'identity\'] or identity != pinned_identity or lot[\'role\'] not in (\'train\',\'val\'):\n        raise ValueError(\'Pinned lot identity/role mismatch\')\n    dataset_root, out = Path(dataset_root), Path(out)\n    if json.loads((dataset_root/\'lot-inventory.json\').read_text()) != lot:\n        raise ValueError(\'Published inventory differs from pinned canonical manifest\')\n    expected = {f[\'path\'] for f in lot[\'files\']}\n    if len(expected) != lot[\'observations\'] or sum(f[\'bytes\'] for f in lot[\'files\']) != lot[\'bytes\']:\n        raise ValueError(\'Manifest counts differ\')\n    if {str(PurePosixPath(p).parent) for p in expected} != set(lot[\'events\']):\n        raise ValueError(\'Manifest event set differs\')\n    for p in expected:\n        if PurePosixPath(p).parts[0] not in (\'2018\',\'2019\',\'2020\'): raise ValueError(\'Held-out year refused\')\n    rasters = dataset_root/\'rasters\'\n    actual = {p.relative_to(rasters).as_posix() for p in rasters.rglob(\'*\') if p.is_file()}\n    if actual != expected: raise ValueError(\'Published raster inventory differs\')\n    out.mkdir(parents=True, exist_ok=False)\n    deadline, checked_bytes = time.monotonic()+seconds, 0\n    with (out/\'verified.jsonl\').open(\'x\', encoding=\'utf-8\') as log:\n        for f in lot[\'files\']:\n            resource_check(deadline)\n            path = safe_child(rasters, f[\'path\'])\n            if path.is_symlink() or path.stat().st_size != f[\'bytes\']: raise ValueError(\'Published size/path differs\')\n            sha, crc, n = hashlib.sha256(), 0, 0\n            with path.open(\'rb\') as fp:\n                for block in iter(lambda: fp.read(BLOCK), b\'\'):\n                    resource_check(deadline)\n                    sha.update(block); crc=zlib.crc32(block,crc); n+=len(block)\n            if n != f[\'bytes\'] or sha.hexdigest() != f[\'sha256\'] or f\'{crc & 0xffffffff:08x}\' != f[\'crc32\']:\n                raise ValueError(\'Published SHA256/CRC differs: \'+f[\'path\'])\n            checked_bytes += n\n            log.write(json.dumps(dict(path=f[\'path\'], sha256=f[\'sha256\'], bytes=n))+\'\\n\'); log.flush()\n    geotiff = None\n    if inspect_geotiff:\n        import rasterio\n        from rasterio.windows import Window\n        with rasterio.Env(GDAL_CACHEMAX=64*1024**2, GDAL_NUM_THREADS=\'1\'):\n            with rasterio.open(safe_child(rasters,lot[\'files\'][0][\'path\'])) as src:\n                tile = src.read(1, window=Window(0,0,min(64,src.width),min(64,src.height)), masked=True)\n                if tile.size == 0: raise ValueError(\'Empty GeoTIFF window\')\n                geotiff = dict(bands=src.count, width=src.width, height=src.height, tile_shape=list(tile.shape), crs=str(src.crs))\n    result = dict(status=\'ONE_LOT_REMOTE_BYTES_CERTIFIED\', handle=lot[\'handle\'], identity=identity,\n                  files=len(expected), events=len(lot[\'events\']), bytes=checked_bytes, role=lot[\'role\'], geotiff=geotiff,\n                  claim=\'This receipt certifies only this attached lot; no training or other lots were validated\')\n    with (out/\'certificate.json\').open(\'x\',encoding=\'utf-8\') as f: json.dump(result,f,indent=2)\n    return result\n', 'kaggle_certify_inputs.py', 'exec'), namespace)
work=namespace['bootstrap_code']('/kaggle/input/geofirex-expanded-code-recovery-v1', '/kaggle/working/geofirex-reviewed', '7f942c6cccebe17096064f3a641f1b110a5048d48a73348bcc1cf6dc0aeeb9fe', '45cc69b9793ea7a85ef6d89eb8ab4416d2e6403e350b04e238781e0ed13a0b77')
runtime=namespace['check_runtime'](work, 'reports/recovery_20261107_v3/finalized/deployment/bundle')
Path('/kaggle/working/runtime-certificate.json').write_text(json.dumps(runtime,indent=2))
print(json.dumps(runtime,indent=2))
certificate=namespace['certify_lot'](work/'reports/recovery_20261107_v3/finalized/deployment/bundle'/'metadata/lot-005.json', '/kaggle/input/geofirex-wsts-fold0-tv-005-v1', '/kaggle/working/lot-005-certificate', pinned_identity='ef579d40d24107bbb034ab4e0cef12a256f7ecf0ddd85f43d5162b6854fd92ff', seconds=3600)
print(json.dumps(certificate,indent=2))
